# Qwen3.5 CPU MWE

**MWE** bedeutet **Minimal Working Example**: das kleinstmoegliche lauffaehige Beispiel fuer genau den relevanten Fall.

## Ziel

Dieses Notebook soll zeigen, wie `Qwen/Qwen3.5-0.8B` auf diesem Rechner **moeglichst einfach und robust auf CPU** laeuft, so dass man denselben Weg spaeter leicht in andere Python-Pakete uebernehmen kann.

## Entscheidung

Wir nehmen hier **Hugging Face / Transformers** als Default.

Warum:
- Diese Route laeuft in `qwen35_stable` bereits stabil.
- Sie ist in Python am leichtesten integrierbar.
- Die Modell-ID kann spaeter sehr leicht gegen ein anderes Modell ausgetauscht werden.

Warum nicht GGUF als Default:
- GGUF war lokal als separate CPU-Route interessant.
- Der getestete Python-Weg mit `llama-cpp-python` war fuer dieses Qwen3.5-GGUF aber noch kein sauberer Notebook-Default.
- Ein neueres externes `llama.cpp` war schneller, ist aber noch nicht die robusteste allgemeine Package-Integration.

## Was war am schnellsten?
- Der schnellste beobachtete lokale CPU-Weg war ein neueres externes `llama.cpp` mit Qwen3.5-GGUF.
- Der **robusteste und einfachste Python-Default** bleibt fuer dieses Notebook trotzdem Hugging Face / Transformers.
- Im HF-Pfad war `4` CPU-Threads auf diesem Rechner der beste einfache Wert.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3.5-0.8B"
CPU_THREADS = 4

torch.set_num_threads(CPU_THREADS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="cpu",
)
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id

def label_topic(keywords: str, representative_docs: str, max_new_tokens: int = 8) -> dict:
    messages = [
        {"role": "system", "content": "You label research topics with 1 to 3 words only."},
        {
            "role": "user",
            "content": (
                "Label this topic in 1-3 words only. "
                f"Topic keywords: {keywords}. "
                f"Representative docs: {representative_docs}."
            ),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([text], add_special_tokens=False, return_tensors="pt")

    t0 = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    t1 = time.perf_counter()

    generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], outputs)
    ]
    label = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return {
        "label": label,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "generate_s": round(t1 - t0, 2),
    }

sample = label_topic(
    keywords="telescope, orbit, spacecraft, galaxy, mission",
    representative_docs="observations of distant galaxies, orbital dynamics, and deep-space missions",
)

print("model_id:", MODEL_ID)
print("cpu_threads:", CPU_THREADS)
print("label:", sample["label"])
print("input_tokens:", sample["input_tokens"])
print("generate_s:", sample["generate_s"])